# Notebook 02: `Voxel51/WLASL` Top-100 Feature Extraction (Visual + Language)

**Runtime:** Google Colab **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`)  
**Persistent Drive Root:** `/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100/`  
**Fast Local Scratch:** `/content/wlasl100_cache/`  

This notebook:
1. Unpacks `videos/voxel51_wlasl100_videos.tar` (`1,120` clips) from Drive onto local Colab SSD (`/content/wlasl100_cache/videos/`) in seconds.
2. Runs frozen **`ResNet50` (`ResNet50_Weights.DEFAULT`, AMP)** over `32` uniformly sampled `224x224` RGB frames per clip, generating `(N, 32, 2048)` `float16` visual features (`val`, `test`, `train`).
3. Encodes all `100` ASL glosses via **`sentence-transformers/all-MiniLM-L6-v2`** (`384-D`) in both raw gloss (`"{gloss}"`) and ASL prompt (`"An American Sign Language sign representing the word '{gloss}'"`) variants.
4. Verifies `0` non-finite/zero rows, `100%` manifest/label parity, and semantic nearest neighbors in text space, persisting all caches to Drive `features/`.


## 1. Environment Setup, T4 GPU Check & Google Drive Mount


In [ ]:
import importlib
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "cv2": "opencv-python-headless",
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "tqdm": "tqdm",
}
missing = []
for mod_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(mod_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import contextlib
import gc
import json
import random
import shutil
import tarfile
import time
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

if IN_COLAB and colab_drive is not None:
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted at /content/drive.")
    else:
        colab_drive.mount("/content/drive")

IS_KAGGLE = Path("/kaggle/working").exists()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python  : {sys.version.split()[0]} | IN_COLAB={IN_COLAB} | IS_KAGGLE={IS_KAGGLE}")
print(f"PyTorch : {torch.__version__} | Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU — switch runtime to GPU for fast extraction.")

## 2. Configuration & Fast SSD Video Staging


In [ ]:
# ==============================================================================
# CONFIGURATION & KAGGLE / COLAB PATH AUTO-DISCOVERY
# ==============================================================================
SEED = 42
NUM_FRAMES = 32
IMAGE_SIZE = 224
VISUAL_DIM = 2048
FEATURE_DTYPE = np.float16
VIDEO_BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
FRAME_CHUNK_SIZE = 128 if DEVICE.type == "cuda" else 32
FLUSH_EVERY_CLIPS = 64 if DEVICE.type == "cuda" else 32
USE_AMP = DEVICE.type == "cuda"

LANG_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PROMPT_TEMPLATE = "An American Sign Language sign representing the word '{gloss}'"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def _find_source_root() -> Path:
    candidates = [
        Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100"),
        Path("/kaggle/working/voxel51_wlasl100"),
        Path.cwd() / "data" / "voxel51_wlasl100",
    ]
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.is_dir():
        for tar_match in sorted(kaggle_in.rglob("voxel51_wlasl100_videos.tar")):
            candidates.append(tar_match.parent.parent)
    for cand in candidates:
        if (cand / "metadata" / "train.csv").is_file() and (cand / "videos" / "voxel51_wlasl100_videos.tar").is_file():
            return cand
    raise FileNotFoundError(
        "Could not locate voxel51_wlasl100 (metadata/train.csv + videos/voxel51_wlasl100_videos.tar). "
        "On Kaggle, attach Notebook 01 output via '+ Add Input -> Notebooks -> Notebook 01'."
    )

SOURCE_ROOT = _find_source_root()
if IN_COLAB and Path("/content/drive/MyDrive").exists():
    OUT_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
    SCRATCH_ROOT = Path("/content/wlasl100_cache")
elif IS_KAGGLE:
    OUT_ROOT = Path("/kaggle/working/voxel51_wlasl100")
    SCRATCH_ROOT = Path("/tmp/wlasl100_cache")
else:
    OUT_ROOT = SOURCE_ROOT
    SCRATCH_ROOT = Path.cwd() / "data" / "wlasl100_scratch"

META_SRC_DIR = SOURCE_ROOT / "metadata"
TAR_PATH = SOURCE_ROOT / "videos" / "voxel51_wlasl100_videos.tar"

META_OUT_DIR = OUT_ROOT / "metadata"
FEATURES_DIR = OUT_ROOT / "features"
EXP_DIR = OUT_ROOT / "experiments"
REPORTS_DIR = OUT_ROOT / "reports"
for d in (META_OUT_DIR, FEATURES_DIR, EXP_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

if META_OUT_DIR.resolve() != META_SRC_DIR.resolve():
    for fname in ("train.csv", "val.csv", "test.csv", "voxel51_wlasl100_master.csv", "class_to_id.json", "id_to_class.json", "signer_split_audit.json"):
        src_f = META_SRC_DIR / fname
        if src_f.is_file():
            shutil.copy2(src_f, META_OUT_DIR / fname)
    src_exp = SOURCE_ROOT / "experiments" / "results_summary.csv"
    if src_exp.is_file() and not (EXP_DIR / "results_summary.csv").is_file():
        shutil.copy2(src_exp, EXP_DIR / "results_summary.csv")

META_DIR = META_SRC_DIR
LOCAL_VIDEOS = SCRATCH_ROOT / "videos"
LOCAL_FEATURES = SCRATCH_ROOT / "features"
LOCAL_VIDEOS.mkdir(parents=True, exist_ok=True)
LOCAL_FEATURES.mkdir(parents=True, exist_ok=True)

print("Input Source Root :", SOURCE_ROOT)
print("Output Save Root  :", OUT_ROOT)
print("Local SSD Scratch :", SCRATCH_ROOT)

t0 = time.time()
existing_mp4s = len(list(LOCAL_VIDEOS.glob("*.mp4")))
if existing_mp4s < 1120:
    print(f"Unpacking {TAR_PATH} -> {LOCAL_VIDEOS} ...")
    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(LOCAL_VIDEOS)
    existing_mp4s = len(list(LOCAL_VIDEOS.glob("*.mp4")))
print(f"Local SSD video count: {existing_mp4s} files (staged in {time.time() - t0:.1f}s)")

## 3. Frozen ResNet50 Visual Feature Extraction (`val`, `test`, `train`)


In [ ]:
# ==============================================================================
# STEP 1: LOAD VALIDATED SPLITS & EXTRACT FROZEN RESNET50 FEATURES (2048-D)
# ==============================================================================
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 1, 3)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 1, 3)

class_to_id = {str(k): int(v) for k, v in json.loads((META_DIR / "class_to_id.json").read_text(encoding="utf-8")).items()}
id_to_class = {int(k): str(v) for k, v in json.loads((META_DIR / "id_to_class.json").read_text(encoding="utf-8")).items()}

def normalize_video_id(val: Any) -> str:
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return str(int(s)) if s.isdigit() else s


splits = {}
for sp in ("train", "val", "test"):
    df = pd.read_csv(META_DIR / f"{sp}.csv")
    df["video_id_str"] = df["video_id"].map(normalize_video_id)
    splits[sp] = df
    print(f"{sp:5s}: {len(df)} clips | {df['class_id'].nunique()} classes | {df['signer_id'].nunique()} signers")


def preprocess_pretrimmed_clip(video_path: Path, num_frames: int = NUM_FRAMES, image_size: int = IMAGE_SIZE) -> tuple[torch.Tensor, int]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    frames_bgr = []
    try:
        while True:
            ok, frm = cap.read()
            if not ok or frm is None:
                break
            frames_bgr.append(frm)
    finally:
        cap.release()

    if not frames_bgr:
        raise RuntimeError(f"Decoded 0 frames: {video_path}")

    indices = np.round(np.linspace(0, len(frames_bgr) - 1, num_frames)).astype(np.int64)
    sampled_rgb = []
    for idx in indices.tolist():
        resized = cv2.resize(frames_bgr[idx], (image_size, image_size), interpolation=cv2.INTER_AREA)
        sampled_rgb.append(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))

    arr = np.stack(sampled_rgb, axis=0).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    arr = np.transpose(arr, (0, 3, 1, 2)).astype(np.float32, copy=False)
    return torch.from_numpy(arr), len(frames_bgr)


weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False


@torch.inference_mode()
def encode_batch(video_tensors: list[torch.Tensor]) -> np.ndarray:
    batch = torch.stack(video_tensors, dim=0).to(DEVICE, non_blocking=True)
    b, t, c, h, w = batch.shape
    frames = batch.reshape(b * t, c, h, w)
    chunks = []
    amp_ctx = torch.amp.autocast("cuda", enabled=USE_AMP) if USE_AMP else contextlib.nullcontext()
    with amp_ctx:
        for pos in range(0, len(frames), FRAME_CHUNK_SIZE):
            feats = backbone(frames[pos : pos + FRAME_CHUNK_SIZE]).flatten(1)
            chunks.append(feats)
    return torch.cat(chunks, dim=0).reshape(b, t, VISUAL_DIM).float().cpu().numpy().astype(FEATURE_DTYPE)


def _sync_split_to_drive(sp: str, x_arr: np.ndarray, y_arr: np.ndarray, done_arr: np.ndarray, man_df: pd.DataFrame) -> None:
    np.save(LOCAL_FEATURES / f"X_{sp}.npy", x_arr)
    np.save(LOCAL_FEATURES / f"y_{sp}.npy", y_arr)
    np.save(LOCAL_FEATURES / f"done_{sp}.npy", done_arr)
    man_df.to_csv(LOCAL_FEATURES / f"manifest_{sp}.csv", index=False)
    for fn in (f"X_{sp}.npy", f"y_{sp}.npy", f"done_{sp}.npy", f"manifest_{sp}.csv"):
        shutil.copy2(LOCAL_FEATURES / fn, FEATURES_DIR / fn)


extraction_stats = []
for sp in ("val", "test", "train"):
    t_sp = time.time()
    df = splits[sp]
    n_rows = len(df)
    y_arr = df["class_id"].to_numpy(dtype=np.int64)

    x_drv = FEATURES_DIR / f"X_{sp}.npy"
    d_drv = FEATURES_DIR / f"done_{sp}.npy"
    m_drv = FEATURES_DIR / f"manifest_{sp}.csv"
    if x_drv.is_file() and d_drv.is_file() and m_drv.is_file():
        x_prev = np.load(x_drv)
        d_prev = np.load(d_drv)
        m_prev = pd.read_csv(m_drv)
        if x_prev.shape == (n_rows, NUM_FRAMES, VISUAL_DIM) and len(d_prev) == n_rows and len(m_prev) == n_rows:
            x_arr = np.array(x_prev, dtype=FEATURE_DTYPE, copy=True)
            done_arr = np.array(d_prev, dtype=bool, copy=True)
            decoded_counts = m_prev["decoded_frames"].to_numpy(dtype=np.int64).copy()
            print(f"Resuming {sp}: {int(done_arr.sum())}/{n_rows} already extracted on Drive.")
        else:
            x_arr = np.zeros((n_rows, NUM_FRAMES, VISUAL_DIM), dtype=FEATURE_DTYPE)
            done_arr = np.zeros(n_rows, dtype=bool)
            decoded_counts = np.zeros(n_rows, dtype=np.int64)
    else:
        x_arr = np.zeros((n_rows, NUM_FRAMES, VISUAL_DIM), dtype=FEATURE_DTYPE)
        done_arr = np.zeros(n_rows, dtype=bool)
        decoded_counts = np.zeros(n_rows, dtype=np.int64)

    man_df = pd.DataFrame({
        "cache_index": np.arange(n_rows, dtype=np.int64),
        "video_id": df["video_id_str"].tolist(),
        "label": y_arr.tolist(),
        "gloss": df["gloss"].astype(str).tolist(),
        "signer_id": df["signer_id"].astype(str).tolist(),
        "decoded_frames": decoded_counts.tolist(),
    })

    pending_idxs = np.flatnonzero(~done_arr)
    batch_tensors, batch_idxs = [], []
    newly_done = 0
    for pos_i, idx in enumerate(tqdm(pending_idxs.tolist(), desc=f"ResNet50 [{sp}]")):
        vid_str = str(df.iloc[idx]["video_id_str"])
        vp = LOCAL_VIDEOS / f"{vid_str}.mp4"
        clip_t, n_dec = preprocess_pretrimmed_clip(vp)
        decoded_counts[idx] = n_dec
        man_df.at[idx, "decoded_frames"] = int(n_dec)
        batch_tensors.append(clip_t)
        batch_idxs.append(idx)

        is_last = pos_i == len(pending_idxs) - 1
        if len(batch_tensors) == VIDEO_BATCH_SIZE or is_last:
            encoded = encode_batch(batch_tensors)
            for bi, target_idx in enumerate(batch_idxs):
                x_arr[target_idx] = encoded[bi]
                done_arr[target_idx] = True
                newly_done += 1
            batch_tensors.clear()
            batch_idxs.clear()

            if newly_done >= FLUSH_EVERY_CLIPS or is_last:
                _sync_split_to_drive(sp, x_arr, y_arr, done_arr, man_df)
                newly_done = 0

    assert done_arr.all(), f"{sp}: incomplete rows!"
    assert np.isfinite(x_arr).all(), f"{sp}: NaN/Inf detected!"
    zero_rows = int((np.abs(x_arr).sum(axis=(1, 2)) == 0).sum())
    assert zero_rows == 0, f"{sp}: zero rows detected: {zero_rows}"
    _sync_split_to_drive(sp, x_arr, y_arr, done_arr, man_df)

    elapsed = round(time.time() - t_sp, 2)
    extraction_stats.append({
        "split": sp,
        "shape": list(x_arr.shape),
        "dtype": str(x_arr.dtype),
        "completed": int(done_arr.sum()),
        "failed": int((~done_arr).sum()),
        "nonfinite": 0,
        "zero_rows": zero_rows,
        "elapsed_sec": elapsed,
    })
    print(f"Finished {sp:5s}: shape={x_arr.shape}, min_frames={decoded_counts.min()}, max_frames={decoded_counts.max()}, elapsed={elapsed}s")

del backbone, resnet
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Pretrained Language Embeddings (`all-MiniLM-L6-v2`)


In [ ]:
# ==============================================================================
# STEP 2: ENCODE 100 ASL GLOSSES WITH PRETRAINED LANGUAGE MODEL (ALL-MINILM-L6-V2)
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(LANG_MODEL_NAME)
text_model = AutoModel.from_pretrained(LANG_MODEL_NAME).to(DEVICE).eval()
for p in text_model.parameters():
    p.requires_grad = False


@torch.inference_mode()
def encode_texts(texts: list[str]) -> torch.Tensor:
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt",
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    out = text_model(**encoded)
    mask = encoded["attention_mask"].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
    pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    return F.normalize(pooled, p=2, dim=-1).cpu().float()


ordered_glosses = [id_to_class[i] for i in range(len(id_to_class))]
gloss_texts_raw = list(ordered_glosses)
gloss_texts_prompt = [PROMPT_TEMPLATE.format(gloss=g) for g in ordered_glosses]

emb_raw = encode_texts(gloss_texts_raw)
emb_prompt = encode_texts(gloss_texts_prompt)

assert emb_raw.shape == (len(ordered_glosses), 384)
assert emb_prompt.shape == (len(ordered_glosses), 384)
assert torch.isfinite(emb_raw).all() and torch.isfinite(emb_prompt).all()

torch.save(
    {
        "text_embeddings": emb_raw,
        "class_to_id": class_to_id,
        "id_to_class": id_to_class,
        "texts": gloss_texts_raw,
        "model_name": LANG_MODEL_NAME,
    },
    FEATURES_DIR / "text_embeddings_gloss.pt",
)
torch.save(
    {
        "text_embeddings": emb_prompt,
        "class_to_id": class_to_id,
        "id_to_class": id_to_class,
        "texts": gloss_texts_prompt,
        "model_name": LANG_MODEL_NAME,
        "prompt_template": PROMPT_TEMPLATE,
    },
    FEATURES_DIR / "text_embeddings_prompt.pt",
)

(FEATURES_DIR / "gloss_texts.json").write_text(
    json.dumps(
        {
            "ordered_glosses": ordered_glosses,
            "raw": gloss_texts_raw,
            "prompt": gloss_texts_prompt,
            "prompt_template": PROMPT_TEMPLATE,
        },
        indent=2,
    ),
    encoding="utf-8",
)

# Semantic similarity sanity check in prompt embedding space
sim = emb_prompt @ emb_prompt.T
print("=== LANGUAGE EMBEDDING SEMANTIC NEIGHBORS (TOP-3 PER SAMPLE GLOSS) ===")
for query in ["drink", "before", "cool", "computer", "help", "mother"]:
    if query not in class_to_id:
        continue
    qid = class_to_id[query]
    vals, idxs = torch.topk(sim[qid], k=4)
    nbrs = [(id_to_class[int(i)], round(float(v), 3)) for v, i in zip(vals[1:], idxs[1:])]
    print(f"  {query:10s} -> {nbrs}")

## 5. Post-Extraction Verification & Summary


In [ ]:
# ==============================================================================
# STEP 3: POST-EXTRACTION VERIFICATION SUMMARY (COPY THIS OUTPUT BACK)
# ==============================================================================
verify_rows = []
for sp in ("train", "val", "test"):
    x_f = np.load(FEATURES_DIR / f"X_{sp}.npy", mmap_mode="r")
    y_f = np.load(FEATURES_DIR / f"y_{sp}.npy")
    d_f = np.load(FEATURES_DIR / f"done_{sp}.npy")
    m_df = pd.read_csv(FEATURES_DIR / f"manifest_{sp}.csv")
    s_df = splits[sp]
    assert x_f.shape == (len(s_df), NUM_FRAMES, VISUAL_DIM)
    assert np.array_equal(y_f, s_df["class_id"].to_numpy(dtype=np.int64))
    assert m_df["video_id"].map(normalize_video_id).tolist() == s_df["video_id_str"].tolist()
    assert bool(d_f.all())
    verify_rows.append({
        "split": sp,
        "shape": str(x_f.shape),
        "dtype": str(x_f.dtype),
        "completed": int(d_f.sum()),
        "manifest_match": True,
        "nonfinite": int((~np.isfinite(x_f)).sum()),
        "mean": round(float(np.asarray(x_f, dtype=np.float32).mean()), 4),
        "std": round(float(np.asarray(x_f, dtype=np.float32).std()), 4),
    })

verify_df = pd.DataFrame(verify_rows)
summary_payload = {
    "dataset": "Voxel51/WLASL Top-100 (Option A)",
    "visual_backbone": "ResNet50_Weights.DEFAULT",
    "language_backbone": LANG_MODEL_NAME,
    "visual_splits": extraction_stats,
    "text_embeddings_shape": list(emb_prompt.shape),
    "text_norm_mean": round(float(torch.norm(emb_prompt, dim=-1).mean()), 5),
}
(FEATURES_DIR / "feature_extraction_summary.json").write_text(
    json.dumps(summary_payload, indent=2), encoding="utf-8"
)

print("=" * 72)
print("NOTEBOOK 02 (VOXEL51 WLASL-100 FEATURE EXTRACTION) COMPLETED")
print("=" * 72)
print(verify_df.to_string(index=False))
print(f"\nLanguage embeddings: gloss={tuple(emb_raw.shape)}, prompt={tuple(emb_prompt.shape)}, L2 norm mean={summary_payload['text_norm_mean']}")
if IS_KAGGLE:
    import zipfile
    zip_out = Path("/kaggle/working/voxel51_wlasl100_features_and_meta.zip")
    with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_STORED) as zf:
        for folder in (FEATURES_DIR, META_OUT_DIR):
            for fp in sorted(folder.glob("*")):
                if fp.is_file():
                    zf.write(fp, arcname=f"voxel51_wlasl100/{folder.name}/{fp.name}")
    print(f"\n[KAGGLE ZIP READY] {zip_out} ({zip_out.stat().st_size / (1024**2):.1f} MB)")

print("\nSaved Files (features/):")
for p in sorted(FEATURES_DIR.glob("*")):
    print(f"  [OK] {p.name:35s} ({p.stat().st_size:,} bytes)")